In [1]:
import pandas as pd
import plotly.express as px
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output

In [2]:
airline_data = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/airline_data.csv',
                           encoding="ISO-8859-1",
                           dtype={'Div1Airport': str, 'Div1TailNum': str, 
                                  'Div2Airport': str, 'Div2TailNum': str})

data = airline_data.sample(n=500, random_state=42)

In [3]:
app = dash.Dash(__name__)

In [4]:
title = html.H1('Flight Details Statistics Dashboard', style={'textAlign': 'center', 'color': '#503D36', 'font-size': 35})
input_box = html.Div(["Input Year: ", dcc.Input(id='input-year', value='2010', type='number', style={'height':'35px', 'font-size': 30})], style={'font-size': 30})

In [5]:
segment1 = html.Div([
    html.Div(dcc.Graph(id='carrier-plot'), style={'width': '50%'}),
    html.Div(dcc.Graph(id='weather-plot'), style={'width': '50%'})
], style={'display': 'flex'})

In [6]:
segment2 = html.Div([
    html.Div(dcc.Graph(id='nas-plot'), style={'width': '50%'}),
    html.Div(dcc.Graph(id='security-plot'), style={'width': '50%'})
], style={'display': 'flex'})

In [7]:
segment3 = html.Div(dcc.Graph(id='late-plot'), style={'width': '100%'})

In [8]:
app.layout = html.Div(children=[
    title,
    input_box,
    html.Br(),
    segment1,
    segment2,
    segment3
])

In [9]:
test_year = 2010
df_test = airline_data[airline_data['Year'] == int(test_year)]
df_test.head()

,Unnamed: 0,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,...,Div4WheelsOff,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum
38,291296,2010,3,8,22,7,2010-08-22,DL,19790,DL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
52,1870601,2010,3,8,31,2,2010-08-31,AA,19805,AA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
66,335161,2010,1,2,16,2,2010-02-16,WN,19393,WN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
70,906836,2010,3,7,28,3,2010-07-28,B6,20409,B6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
76,1041464,2010,1,3,15,1,2010-03-15,EV,20366,EV,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
test_carrier = df_test.groupby(['Reporting_Airline'])['CarrierDelay'].mean().reset_index()
test_carrier.head()

,Reporting_Airline,CarrierDelay
0,9E,5.285714
1,AA,26.888889
2,AS,0.000000
3,B6,22.000000
4,CO,0.250000


In [11]:
def compute_data(annual_data, entered_year):
    df = annual_data[annual_data['Year'] == int(entered_year)]
    
    avg_car = df.groupby(['Reporting_Airline'])['CarrierDelay'].mean().reset_index()
    avg_weather = df.groupby(['Reporting_Airline'])['WeatherDelay'].mean().reset_index()
    avg_NAS = df.groupby(['Reporting_Airline'])['NASDelay'].mean().reset_index()
    avg_sec = df.groupby(['Reporting_Airline'])['SecurityDelay'].mean().reset_index()
    avg_late = df.groupby(['Reporting_Airline'])['LateAircraftDelay'].mean().reset_index()
    
    return avg_car, avg_weather, avg_NAS, avg_sec, avg_late

In [12]:
@app.callback(
    [Output('carrier-plot', 'figure'), Output('weather-plot', 'figure'), 
     Output('nas-plot', 'figure'), Output('security-plot', 'figure'), 
     Output('late-plot', 'figure')],
    [Input('input-year', 'value')]
)
def get_graph(entered_year):
    data_list = compute_data(airline_data, entered_year)
    cols = ['CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']
    titles = ['carrier', 'weather', 'NAS', 'security', 'late aircraft']
    figs = []
    
    for i in range(5):
        figs.append(px.line(data_list[i], x='Reporting_Airline', y=cols[i], color='Reporting_Airline', title=f'Average {titles[i]} delay time by airline'))
        
    return figs

In [13]:
app.run(debug=True, use_reloader=False, jupyter_mode="tab", port=8055)

Dash app running on http://127.0.0.1:8055/


<IPython.core.display.Javascript object>

In [15]:
app.run(debug=False, use_reloader=False, jupyter_mode="tab", port=8057)

Dash app running on http://127.0.0.1:8057/


<IPython.core.display.Javascript object>